# Diabetes Classification: Exploratory Analysis and Model Evaluation

This notebook develops a complete, reproducible binary-classification workflow for the **Diabetes Classification** dataset supplied with the original exercise.

### Objectives
1. Perform descriptive statistics and exploratory data analysis (EDA).
2. Prepare categorical variables using one-hot encoding.
3. Select informative features using `SelectKBest` and ANOVA F-scores.
4. Train and evaluate **K-Nearest Neighbours (KNN)** and **Support Vector Machine (SVM)** classifiers.
5. Fix the ROC-curve error in the original notebook.
6. Complete the assignment: confusion matrices and ROC curves for the models, explain evaluation metrics, compare model performance graphically, and repeat the analysis using label encoding.

> **Note:** This is an educational machine-learning exercise, not a clinical diagnostic system. Model predictions should not be used for medical decisions.

## 1. Import libraries

We use pandas and NumPy for data manipulation, Matplotlib/Seaborn for visualisation, and scikit-learn for preprocessing, feature selection, modelling and evaluation.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 10
TEST_SIZE = 0.40

## 2. Load the dataset

The original notebook used an absolute macOS path. That makes the notebook difficult to run on another computer. The repository therefore expects the CSV at `data/Diabetes Classification.csv`.

If your CSV has a different name or location, change `DATA_PATH` below.

In [ ]:
DATA_PATH = Path("../data/Diabetes Classification.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. "
        "Place 'Diabetes Classification.csv' inside the repository's data/ folder."
    )

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
df.head()

## 3. Initial data inspection

The dataset contains demographic, physiological and lifestyle variables, with `Diagnosis` as the binary target. We first inspect data types, missing values and duplicate records before modelling.

In [ ]:
df.info()

In [ ]:
print("Missing values by column:")
display(df.isnull().sum().to_frame("missing_values"))

print(f"Duplicate rows: {df.duplicated().sum()}")
print("\nTarget distribution:")
display(df["Diagnosis"].value_counts().to_frame("count"))

## 4. Descriptive analysis

Descriptive statistics help us understand the scale and distribution of numerical variables before modelling. For categorical variables, frequency counts are more appropriate.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

display(df[numeric_cols].describe().T)

In [ ]:
for col in categorical_cols:
    print(f"\n{col}")
    display(df[col].value_counts(dropna=False).to_frame("count"))

## 5. Exploratory data analysis

EDA is used to look for class imbalance, unusual distributions and simple relationships between predictors and the target. These plots describe the dataset; they do not establish causal relationships.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x="Diagnosis", ax=axes[0])
axes[0].set_title("Diagnosis distribution")
axes[0].set_xlabel("Diagnosis")

if numeric_cols:
    sns.boxplot(data=df, x="Diagnosis", y=numeric_cols[0], ax=axes[1])
    axes[1].set_title(f"{numeric_cols[0]} by diagnosis")

plt.tight_layout()
plt.show()

In [ ]:
if len(numeric_cols) > 1:
    plt.figure(figsize=(9, 6))
    sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Correlation matrix of numerical variables")
    plt.tight_layout()
    plt.show()

## 6. Encode categorical predictors

The original notebook correctly recognised that machine-learning estimators need numerical input. For the primary analysis, nominal categorical variables are **one-hot encoded**, so categories such as `Male` and `Female` do not receive an artificial numeric order.

The target `Diagnosis` is encoded separately as 0/1.

In [ ]:
target_col = "Diagnosis"
X_raw = df.drop(columns=target_col)
y_raw = df[target_col].copy()

# Encode the binary target only.
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y_raw)
print("Target mapping:", dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_))))

feature_categorical = X_raw.select_dtypes(exclude=np.number).columns.tolist()
feature_numeric = X_raw.select_dtypes(include=np.number).columns.tolist()

preprocessor_ohe = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), feature_numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop=None), feature_categorical),
    ]
)

## 7. Train/test split

We hold out 40% of the observations for testing, matching the split size used in the original exercise. `stratify=y` keeps the class proportions approximately consistent between the training and test sets, which is particularly useful for a small classification dataset.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print("Training rows:", len(X_train_raw))
print("Testing rows:", len(X_test_raw))

## 8. Feature selection

`SelectKBest` with `f_classif` ranks predictors according to their ANOVA F-statistic against the binary target. We use the top five transformed features, as in the original notebook.

Feature selection must be fitted on the training data only to avoid leaking information from the test set.

In [ ]:
# Fit preprocessing on training data only.
X_train_encoded = preprocessor_ohe.fit_transform(X_train_raw)
X_test_encoded = preprocessor_ohe.transform(X_test_raw)

feature_names = preprocessor_ohe.get_feature_names_out()
selector = SelectKBest(score_func=f_classif, k=min(5, X_train_encoded.shape[1]))
X_train_selected = selector.fit_transform(X_train_encoded, y_train)
X_test_selected = selector.transform(X_test_encoded)

selected_features = feature_names[selector.get_support()]
print("Selected features:")
for feature in selected_features:
    print("-", feature)

## 9. Train KNN and SVM models

KNN is distance-based, so scaling is important. SVM can also be sensitive to feature scale, particularly with an RBF kernel. The preprocessing above standardises numerical variables before modelling and converts categorical variables to numerical indicators.

In [ ]:
models = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)
}

results = {}
predictions = {}
probabilities = {}

for name, model in models.items():
    model.fit(X_train_encoded, y_train)
    pred = model.predict(X_test_encoded)
    proba = model.predict_proba(X_test_encoded)[:, 1]
    predictions[name] = pred
    probabilities[name] = proba
    results[name] = {
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1 Score": f1_score(y_test, pred, zero_division=0),
        "ROC AUC": roc_auc_score(y_test, proba)
    }

results_df = pd.DataFrame(results).T
results_df

## 10. Classification reports

A classification report provides precision, recall and F1-score for each class. Because the dataset is small and the classes may not be perfectly balanced, accuracy should not be interpreted in isolation.

In [ ]:
for name, pred in predictions.items():
    print(f"\n{name} classification report")
    print(classification_report(y_test, pred, zero_division=0))

## 11. Confusion matrices — assignment task 1

A confusion matrix separates predictions into true negatives, false positives, false negatives and true positives. This lets us see which class the classifier is struggling to identify.

The original notebook already plotted an SVM confusion matrix; here we complete the task for **both KNN and SVM**.

In [ ]:
for name, pred in predictions.items():
    cm = confusion_matrix(y_test, pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.show()

## 12. ROC curves and AUC — assignment task 1

The **ROC curve** shows the trade-off between true-positive rate (sensitivity/recall) and false-positive rate across classification thresholds.

**AUC (Area Under the ROC Curve)** summarises the ROC curve into a single value: higher values generally indicate better ranking ability. An AUC of 0.5 corresponds to random ranking, while 1.0 represents perfect separation.

The original ROC cell failed because the f-string used nested braces incorrectly: `f'{'svm'} ...'`. The corrected approach is simply `f'SVM (AUC = {auc:.2f})'`.

In [ ]:
plt.figure(figsize=(7, 5))

for name, proba in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.2f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — One-Hot Encoded Features")
plt.legend()
plt.tight_layout()
plt.show()

## 13. Evaluation metrics explained — assignment task 2

For binary classification, let:
- **TP** = true positives
- **TN** = true negatives
- **FP** = false positives
- **FN** = false negatives

**Accuracy** = `(TP + TN) / (TP + TN + FP + FN)`

Measures the proportion of all predictions that are correct. It can be misleading when classes are imbalanced.

**Precision** = `TP / (TP + FP)`

Of the observations predicted as positive, precision tells us how many were actually positive.

**Recall** = `TP / (TP + FN)`

Of the actual positive observations, recall tells us how many the model detected.

**F1-score** = harmonic mean of precision and recall. It is useful when both false positives and false negatives matter.

**ROC AUC**

Measures how well the model ranks positive cases above negative cases across thresholds. It is threshold-independent, unlike accuracy, precision and recall at one chosen threshold.

## 14. Compare KNN and SVM graphically — assignment task 3

In [ ]:
plot_df = results_df[["Accuracy", "Precision", "Recall", "F1 Score", "ROC AUC"]]
ax = plot_df.plot(kind="bar", figsize=(10, 5))
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("KNN vs SVM — Model Performance")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 15. Re-do the analysis using label encoding — assignment task 4

For this second experiment, every categorical predictor is converted to an integer label. This is deliberately done to satisfy the assignment and allow a direct comparison.

A key caveat is that label encoding introduces an arbitrary numerical ordering. For nominal variables such as gender, smoking status or diet category, that ordering has no real-world meaning. Distance-based KNN can therefore be particularly sensitive to this representation.

In [ ]:
X_label = X_raw.copy()
label_encoders = {}

for col in X_label.select_dtypes(exclude=np.number).columns:
    encoder = LabelEncoder()
    X_label[col] = encoder.fit_transform(X_label[col].astype(str))
    label_encoders[col] = encoder

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_label, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
Xl_train_scaled = scaler.fit_transform(Xl_train)
Xl_test_scaled = scaler.transform(Xl_test)

selector_label = SelectKBest(score_func=f_classif, k=min(5, Xl_train_scaled.shape[1]))
Xl_train_selected = selector_label.fit_transform(Xl_train_scaled, yl_train)
Xl_test_selected = selector_label.transform(Xl_test_scaled)

label_selected_features = X_label.columns[selector_label.get_support()]
print("Top five features after label encoding:")
print(list(label_selected_features))

In [ ]:
label_models = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)
}

label_results = {}
label_predictions = {}
label_probabilities = {}

for name, model in label_models.items():
    model.fit(Xl_train_scaled, yl_train)
    pred = model.predict(Xl_test_scaled)
    proba = model.predict_proba(Xl_test_scaled)[:, 1]
    label_predictions[name] = pred
    label_probabilities[name] = proba
    label_results[name] = {
        "Accuracy": accuracy_score(yl_test, pred),
        "Precision": precision_score(yl_test, pred, zero_division=0),
        "Recall": recall_score(yl_test, pred, zero_division=0),
        "F1 Score": f1_score(yl_test, pred, zero_division=0),
        "ROC AUC": roc_auc_score(yl_test, proba)
    }

label_results_df = pd.DataFrame(label_results).T
label_results_df

## 16. Label-encoded confusion matrices and ROC curves

In [ ]:
for name, pred in label_predictions.items():
    cm = confusion_matrix(yl_test, pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title(f"Confusion Matrix — {name} (Label Encoding)")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.show()

plt.figure(figsize=(7, 5))
for name, proba in label_probabilities.items():
    fpr, tpr, _ = roc_curve(yl_test, proba)
    auc = roc_auc_score(yl_test, proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.2f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Label-Encoded Features")
plt.legend()
plt.tight_layout()
plt.show()

## 17. One-hot vs label encoding — final comparison

This table puts the two encoding strategies side by side. The comparison should focus on the full set of metrics rather than accuracy alone.

In [ ]:
comparison = pd.concat(
    [
        results_df.assign(Encoding="One-Hot"),
        label_results_df.assign(Encoding="Label")
    ]
).reset_index(names="Model")

display(comparison[["Encoding", "Model", "Accuracy", "Precision", "Recall", "F1 Score", "ROC AUC"]])

In [ ]:
metric = "F1 Score"
comparison_plot = comparison.pivot(index="Model", columns="Encoding", values=metric)
ax = comparison_plot.plot(kind="bar", figsize=(8, 5))
ax.set_ylim(0, 1.05)
ax.set_ylabel(metric)
ax.set_title(f"{metric}: One-Hot vs Label Encoding")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 18. Findings and interpretation

The exact numerical conclusions below are generated from the dataset when the notebook is run. Because the CSV was not included with the original notebook, the repository intentionally does not invent missing results.

When interpreting the completed output, consider:
- whether one model has better recall for the positive class;
- whether precision and recall are balanced, as reflected by F1-score;
- whether ROC AUC supports the ranking performance seen in the threshold-based metrics;
- whether feature selection changes performance materially; and
- whether label encoding changes KNN/SVM performance relative to one-hot encoding.

A meaningful difference between encoding strategies would illustrate why representation choice matters: one-hot encoding preserves nominal-category semantics, while label encoding creates artificial numeric distances.

## 19. Key fixes made to the original notebook

1. **ROC curve SyntaxError fixed:** the malformed nested f-string was replaced with a valid f-string.
2. **Selected-feature split fixed:** the original notebook created `x1 = one_encoded[selected_features]` but then accidentally split `x` instead of `x1`.
3. **SVM selected-feature report fixed:** the original report printed `y_pred_svm` twice instead of evaluating `y_pred_svm1`.
4. **Data path made portable:** replaced the original machine-specific absolute path with a repository-relative path.
5. **Scaling added:** numerical variables are standardised because KNN and RBF-SVM are sensitive to feature scale.
6. **Data leakage reduced:** preprocessing and feature selection are fitted on training data only.
7. **Assignment completed:** both models receive confusion matrices and ROC curves; metrics are explained; model performance is plotted; and the analysis is repeated using label encoding.